# 04 - Stage 2 RL Fine-tuning (Colab A100)
PPO-based reinforcement learning on top of the Stage 1 pre-trained REACT model.

## Setup Instructions
Before running this notebook:
1. Stage 1 must be complete — `checkpoint_best.pt` must exist at `MyDrive/CiteMind/checkpoints/pretrain/`
2. `data.zip` must still be in `MyDrive/CiteMind/` (same as Stage 1)
3. Set runtime to **A100 GPU**: Runtime → Change runtime type → A100 GPU
4. Run all cells in order

In [1]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/CiteMind'
DATA_ZIP    = f'{DRIVE_DIR}/data.zip'
PRETRAIN_CKPT_DIR = f'{DRIVE_DIR}/checkpoints/pretrain'
RL_CKPT_DIR = f'{DRIVE_DIR}/checkpoints/rl'

import os
os.makedirs(RL_CKPT_DIR, exist_ok=True)
print('Drive mounted.')
print(f'Stage 1 checkpoint dir : {PRETRAIN_CKPT_DIR}')
print(f'Stage 2 checkpoint dir : {RL_CKPT_DIR}')

Mounted at /content/drive
Drive mounted.
Stage 1 checkpoint dir : /content/drive/MyDrive/CiteMind/checkpoints/pretrain
Stage 2 checkpoint dir : /content/drive/MyDrive/CiteMind/checkpoints/rl


In [2]:
# ── Step 2: Clone / pull repo ────────────────────────────────────────────
import os
if not os.path.exists('/content/repo'):
    !git clone https://github.com/mohamedzait20003/ECE595NLP-Project /content/repo
else:
    !git -C /content/repo pull origin main
%cd /content/repo
print('Repo ready.')

Cloning into '/content/repo'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 257 (delta 159), reused 151 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 7.14 MiB | 19.60 MiB/s, done.
Resolving deltas: 100% (159/159), done.
/content/repo
Repo ready.


In [3]:
# ── Step 3: Install dependencies ─────────────────────────────────────────
!apt-get install -q libsndfile1
!pip install -q -r requirements.txt
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124
!pip install -q sentence-transformers
print('Dependencies installed.')

Reading package lists...
Building dependency tree...
Reading state information...
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Dependencies installed.


In [4]:
# ── Step 4: Extract data ─────────────────────────────────────────────────
import os, json, re
from pathlib import Path

if not os.path.exists(DATA_ZIP):
    raise FileNotFoundError(
        f'Data zip not found at {DATA_ZIP}\n'
        'Please upload data.zip to MyDrive/CiteMind/ in Google Drive.'
    )

print(f'Found: {DATA_ZIP}')
!unzip -q -o "{DATA_ZIP}" -d /content/repo/src/data
print('Zip extracted.')

# Patch Windows absolute paths in manifest JSON files → Colab paths
AUDIO_BASE = '/content/repo/src/data/audio'
target = Path('/content/repo/src/data')

for manifest_name in ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']:
    manifest_path = target / 'audio' / manifest_name
    if not manifest_path.exists():
        continue
    with open(manifest_path, 'r', encoding='utf-8') as f:
        entries = json.load(f)
    patched = 0
    for entry in entries:
        ap = entry.get('audio_path', '')
        if not ap.startswith('/content'):
            parts = re.split(r'[/\\]', ap)
            fname  = parts[-1]
            subdir = parts[-2] if len(parts) >= 2 else manifest_name.split('_')[0]
            entry['audio_path'] = f'{AUDIO_BASE}/{subdir}/{fname}'
            patched += 1
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(entries, f)
    print(f'  Patched {patched} paths in {manifest_name}')

# Verify
print()
for f in ['src/data/audio/train_manifest.json',
          'src/data/audio/val_manifest.json',
          'src/data/processed/train.json']:
    status = 'OK' if os.path.exists(f'/content/repo/{f}') else 'MISSING'
    print(f'  [{status}]  {f}')

Found: /content/drive/MyDrive/CiteMind/data.zip
Zip extracted.
  Patched 15211 paths in train_manifest.json
  Patched 1901 paths in val_manifest.json
  Patched 1902 paths in test_manifest.json

  [OK]  src/data/audio/train_manifest.json
  [OK]  src/data/audio/val_manifest.json
  [OK]  src/data/processed/train.json


In [5]:
# ── Step 5: Verify GPU & Stage 1 checkpoint ──────────────────────────────
import sys, torch
sys.path.insert(0, '/content/repo')

assert torch.cuda.is_available(), 'No GPU found! Set runtime to A100.'
print(f'Device : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import os
stage1_ckpt = f'{PRETRAIN_CKPT_DIR}/checkpoint_best.pt'
assert os.path.exists(stage1_ckpt), f'Stage 1 checkpoint not found: {stage1_ckpt}'
print(f'Stage 1 checkpoint : OK  ({stage1_ckpt})')

Device : NVIDIA A100-SXM4-40GB
VRAM   : 42.4 GB
Stage 1 checkpoint : OK  (/content/drive/MyDrive/CiteMind/checkpoints/pretrain/checkpoint_best.pt)


In [6]:
# ── Step 6: Configure RL training ────────────────────────────────────────
import yaml
from pathlib import Path

config_path = Path('/content/repo/src/config/rl_config.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Point to Drive checkpoint dirs
config['rl']['stage1_checkpoint']   = stage1_ckpt
config['training']['checkpoint_dir'] = RL_CKPT_DIR

# A100 settings
config['training']['fp16']       = True
config['training']['batch_size'] = 4          # RL rollouts use more memory
config['training']['total_steps'] = 5000
config['data']['num_workers']    = 4

with open(config_path, 'w') as f:
    yaml.dump(config, f)

print('RL Config (A100):')
print(f"  stage1_checkpoint : {config['rl']['stage1_checkpoint']}")
print(f"  total_steps       : {config['training']['total_steps']}")
print(f"  batch_size        : {config['training']['batch_size']}")
print(f"  fp16              : {config['training']['fp16']}")
print(f"  checkpoint_dir    : {config['training']['checkpoint_dir']}")
print(f"  reward weights    : {config['rl']['reward_weights']}")

RL Config (A100):
  stage1_checkpoint : /content/drive/MyDrive/CiteMind/checkpoints/pretrain/checkpoint_best.pt
  total_steps       : 5000
  batch_size        : 4
  fp16              : True
  checkpoint_dir    : /content/drive/MyDrive/CiteMind/checkpoints/rl
  reward weights    : {'retrieval': 0.4, 'nli': 0.4, 'hallucination': 0.2}


In [7]:
# ── Step 7: Run RL training ──────────────────────────────────────────────
from src.main.training.rl_train import rl_train
rl_train(str(config_path))

Device: cuda
Rollouts per sample: 4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

RL fine-tuning:   0%|          | 0/5000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x50265 and 768x1)

In [ ]:
# ── Step 8: Plot RL reward curve ─────────────────────────────────────────
import json
import matplotlib.pyplot as plt
from pathlib import Path

log_path = Path(RL_CKPT_DIR) / 'rl_log.json'

with open(log_path, 'r') as f:
    history = json.load(f)

steps   = [h['step']   for h in history]
rewards = [h['reward'] for h in history]
pg_loss = [h['pg_loss'] for h in history]
vf_loss = [h['vf_loss'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, rewards, linewidth=1, color='green')
axes[0].set_title('Combined Reward')
axes[0].set_xlabel('Step')
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, pg_loss, linewidth=1, color='blue')
axes[1].set_title('Policy Gradient Loss')
axes[1].set_xlabel('Step')
axes[1].grid(True, alpha=0.3)

axes[2].plot(steps, vf_loss, linewidth=1, color='orange')
axes[2].set_title('Value Function Loss')
axes[2].set_xlabel('Step')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final reward : {rewards[-1]:.4f}')
print(f'Best  reward : {max(rewards):.4f}')

In [ ]:
# ── Step 9: Sanity check on best RL checkpoint ───────────────────────────
import torch
from pathlib import Path
from src.main.model.main_model import MainModel
from transformers import BartTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt_path = Path(RL_CKPT_DIR) / 'checkpoint_best_rl.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Best RL checkpoint — step: {ckpt["step"]} | best_reward: {ckpt["best_reward"]:.4f}')

model = MainModel(
    whispher_model='openai/whisper-small',
    bart_model='facebook/bart-base'
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

ctx = 'A Survey on Natural Language Processing with Deep Learning'
enc = tokenizer(ctx, return_tensors='pt', max_length=128, truncation=True)

dummy_audio = torch.randn(1, 80, 3000).to(device)
text_ids    = enc['input_ids'].to(device)
text_mask   = enc['attention_mask'].to(device)

with torch.no_grad():
    out = model.generate(
        audio_features=dummy_audio,
        text_input_ids=text_ids,
        text_attention_mask=text_mask,
        max_length=32,
        num_beams=3,
    )

print('Generated:', tokenizer.decode(out[0], skip_special_tokens=True))
print('RL Checkpoints saved to Google Drive at:', RL_CKPT_DIR)